# Phase C0 — ML Data Contract

This notebook establishes the ML data contract before normalization,
EDA, feature selection, train/validation/test splitting, and model training.

It uses the actual CSV files under `data/generated_csv/`.


In [1]:
from pathlib import Path
import json
import pandas as pd

# ============================================================
# PROJECT / DATA PATH SETUP
# ============================================================

CURRENT = Path.cwd()

# The notebook may be executed either:
# 1. from the project root
# 2. from the notebooks/ directory
#
# Find the project root by locating data/generated_csv.

if (CURRENT / "data" / "generated_csv").exists():
    PROJECT_ROOT = CURRENT

elif (CURRENT.parent / "data" / "generated_csv").exists():
    PROJECT_ROOT = CURRENT.parent

else:
    raise FileNotFoundError(
        "Could not locate project root.\n"
        "Expected to find: data/generated_csv\n"
        f"Current working directory: {CURRENT}"
    )

# Define project directories
DATA_DIR = PROJECT_ROOT / "data" / "generated_csv"
REPORT_DIR = PROJECT_ROOT / "reports" / "ml_contract"
DOCS_DIR = PROJECT_ROOT / "docs"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

# Create output directory if needed
REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOCS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

NOTEBOOK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# VERIFY DATA DIRECTORY
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data directory does not exist:\n{DATA_DIR}"
    )

# Show paths so we can verify everything before continuing
print("=" * 60)
print("PROJECT PATH CONFIGURATION")
print("=" * 60)

print(f"Current working directory : {CURRENT}")
print(f"Project root              : {PROJECT_ROOT}")
print(f"Data directory            : {DATA_DIR}")
print(f"Reports directory         : {REPORT_DIR}")
print(f"Docs directory            : {DOCS_DIR}")
print(f"Notebook directory        : {NOTEBOOK_DIR}")

print("\nData directory exists:", DATA_DIR.exists())

# Show available CSV files
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"\nCSV files found: {len(csv_files)}")

for file in csv_files:
    print(f"  - {file.name}")


PROJECT PATH CONFIGURATION
Current working directory : c:\Users\Soham\Documents\revenue-recovery\notebooks
Project root              : c:\Users\Soham\Documents\revenue-recovery
Data directory            : c:\Users\Soham\Documents\revenue-recovery\data\generated_csv
Reports directory         : c:\Users\Soham\Documents\revenue-recovery\reports\ml_contract
Docs directory            : c:\Users\Soham\Documents\revenue-recovery\docs
Notebook directory        : c:\Users\Soham\Documents\revenue-recovery\notebooks

Data directory exists: True

CSV files found: 8
  - action_log.csv
  - case_outcomes.csv
  - customers.csv
  - ml_datamart.csv
  - payment_attempts.csv
  - recovery_cases.csv
  - recovery_events.csv
  - subscriptions.csv


## 1. Dataset Inventory

In [2]:

files = [
    "customers.csv",
    "subscriptions.csv",
    "payment_attempts.csv",
    "recovery_cases.csv",
    "recovery_events.csv",
    "action_log.csv",
    "case_outcomes.csv",
    "ml_datamart.csv",
]

inventory = []

for filename in files:

    path = DATA_DIR / filename

    if not path.exists():
        continue

    df = pd.read_csv(
        path,
        low_memory=False,
    )

    inventory.append(
        {
            "dataset": filename,
            "rows": len(df),
            "columns": len(df.columns),
        }
    )

inventory_df = pd.DataFrame(
    inventory
)

display(inventory_df)


,dataset,rows,columns
0,customers.csv,20000,5
1,subscriptions.csv,22078,12
2,payment_attempts.csv,264936,11
3,recovery_cases.csv,103026,19
4,recovery_events.csv,986365,9
5,action_log.csv,157423,20
6,case_outcomes.csv,103026,17
7,ml_datamart.csv,157423,45


## 2. Exact ML Grain

In [3]:

ml = pd.read_csv(
    DATA_DIR / "ml_datamart.csv",
    low_memory=False,
)

grain = {
    "total_rows": len(ml),
    "unique_cases":
        ml["case_id"].nunique()
        if "case_id" in ml.columns
        else None,

    "unique_customers":
        ml["customer_id"].nunique()
        if "customer_id" in ml.columns
        else None,

    "unique_subscriptions":
        ml["subscription_id"].nunique()
        if "subscription_id" in ml.columns
        else None,

    "unique_policies":
        ml["logging_policy"].nunique()
        if "logging_policy" in ml.columns
        else None,
}

if grain["unique_cases"]:
    grain["avg_rows_per_case"] = (
        grain["total_rows"]
        / grain["unique_cases"]
    )

display(
    pd.DataFrame([grain])
)


,total_rows,unique_cases,unique_customers,unique_subscriptions,unique_policies,avg_rows_per_case
0,157423,25775,10924,11596,3,6.107585



## 3. Target Contract

Dataset V1 uses `y` as the preferred target if present.

`censored` is not used to define success.


In [4]:
target_col = (
    "y"
    if "y" in ml.columns
    else (
        "y_intervention"
        if "y_intervention" in ml.columns
        else None
    )
)

if target_col is None:
    raise ValueError(
        "No target column `y` or `y_intervention` found."
    )

target_raw = ml[target_col]
target = pd.to_numeric(target_raw, errors="coerce")

# C0 contract guard: target must be binary and must not be derived from `censored`.
non_null_target = target.dropna()
invalid_values = sorted(set(non_null_target.unique()) - {0, 1})

if invalid_values:
    raise ValueError(
        f"Target `{target_col}` is not binary 0/1. "
        f"Observed invalid values: {invalid_values}"
    )

missing_target_count = int(target.isna().sum())

print(f"Verified target column: {target_col}")
print(f"Missing target values: {missing_target_count:,}")

if missing_target_count:
    print(
        "NOTE: C0 does not drop or impute missing labels. "
        "C1 will decide how to handle them after auditing."
    )

display(
    pd.DataFrame(
        {
            "target": [target_col],
            "dtype": [str(target.dtype)],
            "unique_values": [target.dropna().unique().tolist()],
            "missing": [missing_target_count],
            "positive_rate": [float(target.mean())],
        }
    )
)

display(
    target.value_counts(
        dropna=False
    ).rename("count").to_frame()
)


Verified target column: y
Missing target values: 0


,target,dtype,unique_values,missing,positive_rate
0,y,int64,"[0, 1]",0,0.228131


,count
y,
0,121510
1,35913


## 4. Complete Column Classification

In [5]:
required_contract_files = [
    "column_classification.csv",
    "candidate_features.csv",
    "excluded_features.csv",
    "feature_availability.csv",
]

missing_contract_files = [
    name for name in required_contract_files
    if not (REPORT_DIR / name).exists()
]

if missing_contract_files:
    raise FileNotFoundError(
        "Missing C0 contract artifacts: "
        + ", ".join(missing_contract_files)
        + ". Run the C0 builder before executing this notebook."
    )

classification = pd.read_csv(
    REPORT_DIR / "column_classification.csv"
)

display(classification)


,column,classification,usable_for_ml,reason,source
0,logging_policy,POLICY / LOGGING,False,Describes the policy that generated/logged the...,ml_datamart
1,case_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
2,subscription_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
3,customer_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
4,decision_seq,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
5,decision_ts_h,TEMPORAL SOURCE FIELD,False,Used to derive decision-time temporal features...,ml_datamart
6,failure_class_at_decision,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
7,first_failure_class,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
8,rail,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart
9,bank_name,OBSERVABLE DECISION-TIME FEATURE,True,Observable case/customer/payment state availab...,ml_datamart



## 5. Candidate Feature Universe

These are candidate inputs only. They are NOT the final model feature set.

Final selection happens after C1 normalization and C2 EDA.


In [6]:

candidates = pd.read_csv(
    REPORT_DIR / "candidate_features.csv"
)

display(candidates)

display(
    candidates
    .groupby("feature_group")
    .size()
    .rename("count")
    .to_frame()
)


,feature_group,column,classification
0,payment_subscription_context,rail,OBSERVABLE DECISION-TIME FEATURE
1,payment_subscription_context,bank_name,OBSERVABLE DECISION-TIME FEATURE
2,payment_subscription_context,segment,OBSERVABLE DECISION-TIME FEATURE
3,payment_subscription_context,plan_tier,OBSERVABLE DECISION-TIME FEATURE
4,payment_subscription_context,amount_inr,OBSERVABLE DECISION-TIME FEATURE
5,payment_subscription_context,value_tier,OBSERVABLE DECISION-TIME FEATURE
6,payment_subscription_context,subscription_age_days,OBSERVABLE DECISION-TIME FEATURE
7,payment_subscription_context,sub_billing_day,OBSERVABLE DECISION-TIME FEATURE
8,current_recovery_state,decision_seq,OBSERVABLE DECISION-TIME FEATURE
9,current_recovery_state,failure_class_at_decision,OBSERVABLE DECISION-TIME FEATURE


,count
feature_group,
current_recovery_state,7
customer_context,4
historical_behaviour,5
intervention_variables,2
payment_subscription_context,8
timing_calendar,6


## 6. Explicit Exclusions

In [7]:

excluded = pd.read_csv(
    REPORT_DIR / "excluded_features.csv"
)

display(excluded)


,column,classification,usable_for_ml,reason,source
0,logging_policy,POLICY / LOGGING,False,Describes the policy that generated/logged the...,ml_datamart
1,case_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
2,subscription_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
3,customer_id,IDENTIFIER,False,Join/group/split identifier. Preserved for dat...,ml_datamart
4,decision_ts_h,TEMPORAL SOURCE FIELD,False,Used to derive decision-time temporal features...,ml_datamart
5,exec_ts_h,POST-DECISION / POSSIBLE LEAKAGE,False,Realized execution or terminal outcome informa...,ml_datamart
6,exec_delay_hours,POST-DECISION / POSSIBLE LEAKAGE,False,Realized execution or terminal outcome informa...,ml_datamart
7,exec_hour_of_day,POST-DECISION / POSSIBLE LEAKAGE,False,Realized execution or terminal outcome informa...,ml_datamart
8,exec_day_of_month,POST-DECISION / POSSIBLE LEAKAGE,False,Realized execution or terminal outcome informa...,ml_datamart
9,exec_in_salary_cluster,POST-DECISION / POSSIBLE LEAKAGE,False,Realized execution or terminal outcome informa...,ml_datamart


## 7. Feature Availability

In [8]:

availability = pd.read_csv(
    REPORT_DIR / "feature_availability.csv"
)

display(availability)


,column,available_at_case_open,available_at_decision,known_only_after_execution,post_outcome,usable_for_ml,classification,reason
0,logging_policy,False,True,False,False,False,POLICY / LOGGING,Describes the policy that generated/logged the...
1,case_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
2,subscription_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
3,customer_id,True,True,False,False,False,IDENTIFIER,Join/group/split identifier. Preserved for dat...
4,decision_seq,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
5,decision_ts_h,False,True,False,False,False,TEMPORAL SOURCE FIELD,Used to derive decision-time temporal features...
6,failure_class_at_decision,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
7,first_failure_class,False,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
8,rail,True,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...
9,bank_name,True,True,False,False,True,OBSERVABLE DECISION-TIME FEATURE,Observable case/customer/payment state availab...



## 8. Normalization Contract

C0 defines normalization requirements only.

No heavy cleaning, outlier removal, imputation, or final feature
selection is performed here.


In [9]:

normalization_rules = {
    "numeric_types":
        "Standardize integer/float representation.",

    "booleans":
        "Normalize boolean-like fields consistently.",

    "categoricals":
        "Normalize whitespace/casing and inspect cardinality.",

    "timestamps":
        "Parse timestamps consistently and derive temporal features.",

    "monetary_units":
        "Confirm monetary fields are INR.",

    "missing_values":
        "Retain missingness for EDA.",

    "outliers":
        "Do not remove/clip before EDA unless structurally invalid.",
}

display(
    pd.DataFrame(
        normalization_rules.items(),
        columns=["area", "rule"],
    )
)


,area,rule
0,numeric_types,Standardize integer/float representation.
1,booleans,Normalize boolean-like fields consistently.
2,categoricals,Normalize whitespace/casing and inspect cardin...
3,timestamps,Parse timestamps consistently and derive tempo...
4,monetary_units,Confirm monetary fields are INR.
5,missing_values,Retain missingness for EDA.
6,outliers,Do not remove/clip before EDA unless structura...



## 9. Contract Outputs

C0 produces:

- `reports/ml_contract/dataset_inventory.csv`
- `reports/ml_contract/column_classification.csv`
- `reports/ml_contract/candidate_features.csv`
- `reports/ml_contract/excluded_features.csv`
- `reports/ml_contract/feature_availability.csv`
- `reports/ml_contract/ml_contract_summary.json`
- `docs/ML_DATA_CONTRACT.md`

### Next

**C1 — normalization + data-quality audit**

then

**C2 — comprehensive EDA**

No model training is performed in C0.
